# Feature Engineering

Prepare the customer support ticket dataset for machine learning by selecting, cleaning, and transforming features.

In [1]:
import pandas as pd

from src.data_loader import load_csv
from src.data_validator import validate_dataframe

In [2]:
df = load_csv("../data/raw/customer_support_tickets.csv")

validate_dataframe(df)

df.head()

2026-07-04 12:48:22,215 | INFO | src.data_loader | Loading dataset: ..\data\raw\customer_support_tickets.csv
2026-07-04 12:48:22,251 | INFO | src.data_loader | Dataset loaded successfully (8469 rows, 17 columns)
2026-07-04 12:48:22,251 | INFO | src.data_validator | Starting dataset validation.
2026-07-04 12:48:22,252 | INFO | src.data_validator | All required columns are present.
2026-07-04 12:48:22,254 | INFO | src.data_validator | No missing values found.
2026-07-04 12:48:22,255 | INFO | src.data_validator | No duplicate Ticket IDs found.
2026-07-04 12:48:22,256 | INFO | src.data_validator | Priority values are valid.
2026-07-04 12:48:22,256 | INFO | src.data_validator | Ticket types are valid.
2026-07-04 12:48:22,257 | INFO | src.data_validator | Ticket status values are valid.
2026-07-04 12:48:22,258 | INFO | src.data_validator | Dataset validation completed successfully.


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## Dataset Shape

In [3]:
df.shape

(8469, 17)

In [4]:
selected_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Customer Satisfaction Rating",
]

In [5]:
df = df[selected_columns]

df.head()

,Ticket Subject,Ticket Description,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased,Customer Satisfaction Rating
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue,Critical,Social media,32,Other,GoPro Hero,NaN
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,Critical,Chat,42,Female,LG Smart TV,NaN
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Low,Social media,48,Other,Dell XPS,3.0
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Low,Social media,27,Female,Microsoft Office,3.0
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Low,Email,67,Female,Autodesk AutoCAD,1.0


## Missing Value Check

Verify that the selected features do not contain missing values before feature engineering.

In [6]:
df.isnull().sum()

Ticket Subject                     0
Ticket Description                 0
Ticket Type                        0
Ticket Priority                    0
Ticket Channel                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Customer Satisfaction Rating    5700
dtype: int64

## Remove Missing Target Values

Remove records that do not contain customer satisfaction ratings since they cannot be used for supervised machine learning.

In [8]:
df = df.dropna(subset=["Customer Satisfaction Rating"])

df.shape

(2769, 9)

In [9]:
df.isnull().sum()

Ticket Subject                  0
Ticket Description              0
Ticket Type                     0
Ticket Priority                 0
Ticket Channel                  0
Customer Age                    0
Customer Gender                 0
Product Purchased               0
Customer Satisfaction Rating    0
dtype: int64

## Encode Categorical Features

Convert categorical variables into numerical values so they can be used by machine learning algorithms.

In [10]:
categorical_columns = [
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "Customer Gender",
    "Product Purchased"
]

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
label_encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

In [13]:
df.head()

,Ticket Subject,Ticket Description,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased,Customer Satisfaction Rating
2,Network problem,I'm facing a problem with my {product_purchase...,4,2,3,48,2,10,3.0
3,Account access,I'm having an issue with the {product_purchase...,0,2,3,27,0,25,3.0
4,Data loss,I'm having an issue with the {product_purchase...,0,2,1,67,0,5,1.0
10,Data loss,I'm having an issue with the {product_purchase...,1,1,2,48,1,30,1.0
11,Software bug,I'm having an issue with the {product_purchase...,2,1,0,51,1,27,1.0


## Verify Data Types

Ensure all features are in the correct numerical format after encoding.

In [14]:
df.dtypes

Ticket Subject                   object
Ticket Description               object
Ticket Type                       int64
Ticket Priority                   int64
Ticket Channel                    int64
Customer Age                      int64
Customer Gender                   int64
Product Purchased                 int64
Customer Satisfaction Rating    float64
dtype: object

## Save Label Encoders

Store the fitted label encoders for future use during inference.

In [15]:
import joblib
import os

In [16]:
os.makedirs("../models", exist_ok=True)

In [17]:
joblib.dump(label_encoders, "../models/label_encoders.pkl")

['../models/label_encoders.pkl']

In [18]:
import os

os.listdir("../models")

['label_encoders.pkl', 'README.md']

## Text Feature Engineering

Convert ticket subject and ticket description into numerical features using TF-IDF vectorization.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
df["Ticket Text"] = (
    df["Ticket Subject"] + " " + df["Ticket Description"]
)

In [21]:
df[["Ticket Subject", "Ticket Description", "Ticket Text"]].head()

,Ticket Subject,Ticket Description,Ticket Text
2,Network problem,I'm facing a problem with my {product_purchase...,Network problem I'm facing a problem with my {...
3,Account access,I'm having an issue with the {product_purchase...,Account access I'm having an issue with the {p...
4,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...
10,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...
11,Software bug,I'm having an issue with the {product_purchase...,Software bug I'm having an issue with the {pro...


In [22]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=500
)

In [23]:
X_text = tfidf.fit_transform(df["Ticket Text"])

In [24]:
X_text.shape

(2769, 500)

In [25]:
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

['../models/tfidf_vectorizer.pkl']

In [26]:
os.listdir("../models")

['label_encoders.pkl', 'README.md', 'tfidf_vectorizer.pkl']

# Train-Test Split

Split the dataset into training and testing sets to evaluate the machine learning model on unseen data.

In [27]:
X = df.drop("Customer Satisfaction Rating", axis=1)
y = df["Customer Satisfaction Rating"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (2769, 9)
Target Shape: (2769,)


# Create Training and Testing Sets

Use an 80-20 split to train the model while keeping a separate testing dataset for evaluation.

In [28]:
from sklearn.model_selection import train_test_split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [30]:
print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels  :", y_test.shape)

Training Features: (2215, 9)
Testing Features : (554, 9)
Training Labels : (2215,)
Testing Labels  : (554,)


# Save Processed Dataset

Save the processed training and testing datasets so they can be used during model development without repeating the feature engineering process.

In [31]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [32]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [33]:
os.listdir("../data/processed")

['README.md', 'X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']

# Feature Engineering Complete

The dataset has been cleaned, prepared, encoded, split into training and testing sets, and saved for model development.